# 03 MLP ??

## ????

- ?????????? MLP ????
- ?? `nn.Sequential` ???????
- ?? logits?label ? `CrossEntropyLoss` ????
- ?? loss ??????????????

## ????

????????????????????????????????????????

MLP ????????????????????????????????????????????????ReLU ??????????????

## ????

- ????????????????????
- ????????????????????????
- Logits?????????????????? softmax?
- Accuracy????????????????

## ??????

??????????????????????? MLP ?????????????

In [ ]:
import torch
from torch import nn
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_moons
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

torch.manual_seed(42)
np.random.seed(42)

x, y = make_moons(n_samples=500, noise=0.18, random_state=42)
x_train, x_test, y_train, y_test = train_test_split(
    x, y, test_size=0.25, random_state=42, stratify=y
)
scaler = StandardScaler()
x_train = scaler.fit_transform(x_train).astype(np.float32)
x_test = scaler.transform(x_test).astype(np.float32)

x_train = torch.tensor(x_train)
y_train = torch.tensor(y_train, dtype=torch.long)
x_test = torch.tensor(x_test)
y_test = torch.tensor(y_test, dtype=torch.long)

plt.figure(figsize=(5, 4))
plt.scatter(x_train[:, 0], x_train[:, 1], c=y_train, cmap="coolwarm", s=18)
plt.title("Training data")
plt.grid(alpha=0.3)
plt.show()

## ????

?????????????????????????????????

### ?????

???? logits ???? `CrossEntropyLoss`??????????????????? label dtype?logits shape????????????????????

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2, 32),
            nn.ReLU(),
            nn.Linear(32, 32),
            nn.ReLU(),
            nn.Linear(32, 2),
        )

    def forward(self, x):
        return self.net(x)

model = MLP()
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
losses = []

for epoch in range(120):
    model.train()
    logits = model(x_train)
    loss = criterion(logits, y_train)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    losses.append(loss.item())

model.eval()
with torch.no_grad():
    test_logits = model(x_test)
    test_pred = test_logits.argmax(dim=1)
    test_acc = (test_pred == y_test).float().mean().item()

print(f"final loss={losses[-1]:.4f}, test accuracy={test_acc:.3f}")

In [ ]:
plt.figure(figsize=(6, 4))
plt.plot(losses)
plt.title("MLP classification loss")
plt.xlabel("Epoch")
plt.ylabel("Cross entropy")
plt.grid(alpha=0.3)
plt.show()

xx, yy = np.meshgrid(
    np.linspace(x_train[:, 0].min() - 0.5, x_train[:, 0].max() + 0.5, 150),
    np.linspace(x_train[:, 1].min() - 0.5, x_train[:, 1].max() + 0.5, 150),
)
grid = torch.tensor(np.c_[xx.ravel(), yy.ravel()].astype(np.float32))
with torch.no_grad():
    zz = model(grid).argmax(dim=1).numpy().reshape(xx.shape)

plt.figure(figsize=(5, 4))
plt.contourf(xx, yy, zz, alpha=0.25, cmap="coolwarm")
plt.scatter(x_test[:, 0], x_test[:, 1], c=y_test, cmap="coolwarm", s=18, edgecolor="k")
plt.title("Decision regions on test points")
plt.grid(alpha=0.3)
plt.show()

## ????

??????????loss ?????????????????????????????????? moon ????????????????????????????

## ????

- ??? label ??? float??? `CrossEntropyLoss` ???
- ? logits ??? softmax???? `CrossEntropyLoss`?
- ???????????????????
- ??????????????????????

?????????? logits ????????? logits ?????????????????

In [ ]:
print("logits shape:", test_logits.shape)
print("label dtype:", y_train.dtype)
print("first logits:", test_logits[:3])

## ????

**Q?????????? ReLU?**  
A??? logits ??????????? `CrossEntropyLoss` ???

**Q??????? softmax ??? `CrossEntropyLoss`?**  
A?`CrossEntropyLoss` ????????????? softmax ????????????????

**Q?????????????????**  
A????????????????????????????????????

## ??

MLP ???????????????????????????? logits?label dtype?loss ?????????????